
# Build a Grounded RAG Chatbot (LangChain)

This notebook follows the lesson sequence and **shows both good and bad patterns** so you can see *why* choices matter.
We’ll start from loading/splitting documents, build a vector index, retrieve and prompt with **grounding**, parse strict JSON, add **guardrails**, optional **tools** and **short memory**, then briefly show **LangSmith**, **serving** (FastAPI + LangServe), and a minimal **UI**.

> **Tip:** This notebook runs even without model/API keys by using a simple **mock LLM** for illustrations. When ready, set real API keys and flip the `USE_REAL_LLM` flag.



## Agenda
1. **Setup & flags** (real vs mock LLM, paths, env)
2. **Loaders & Splitters** (bad vs good chunking; metadata hygiene)
3. **Embeddings & Vector Store** (build, persist, reload)
4. **Retriever** (k, filters; bad vs good choices)
5. **Grounded Prompt → JSON** (bad vs good prompt; temperature)
6. **Parsing & Validation** (retry once; friendly errors)
7. **Guardrails** (ACL filter, PII scrub, context size/auto-compress)
8. **Tools (optional)** (safe calculator, one search tool; routing)
9. **Memory (short & safe)** (`RunnableWithMessageHistory`)
10. **Observe** (LangSmith overview + enable flags)
11. **Serve** (FastAPI + LangServe, contract stays the same)
12. **UI** (tiny Gradio chat that shows citations)


## 1) Setup & flags

In [1]:

# --- Flags ---
USE_REAL_LLM = False   # Set to True to use a real model (requires API keys)
MODEL_NAME = "gpt-4o-mini"  # or another chat model your provider supports

# --- Optional: environment variables (uncomment and set in your env) ---
# import os
# os.environ["OPENAI_API_KEY"] = "..."
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "Module9_RAG"

# --- Install tips (uncomment to install) ---
# %pip install -q langchain langchain-openai langchain-community faiss-cpu chromadb tiktoken gradio fastapi uvicorn langserve



## 2) Loaders & Splitters

We’ll create a **tiny sample corpus** right here so the notebook runs anywhere. In your project, replace this with `DirectoryLoader`, PDF/MD loaders, etc.

### Bad pattern (for teaching):
- One giant chunk (no splitting) → retrieval returns an overlong blob; prompt bloat; poor grounding.

### Good pattern:
- `RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80)`
- Preserve **metadata** including a **`chunk_id`** so we can cite sources.


In [2]:

# Create a small in-memory corpus (simulating PDF/MD content)
sample_docs = [
    {
        "title": "Install Guide",
        "section": "Hardware Requirements",
        "text": "To install, you need 8 GB RAM, 20 GB free disk, and a modern CPU. GPU is optional for local inference. Make sure to update drivers before install."
    },
    {
        "title": "Account FAQ",
        "section": "Password Reset",
        "text": "If you forget your password, use the 'Forgot Password' link on the sign-in page. An email with a reset link will be sent to your registered address."
    },
    {
        "title": "Support Handbook",
        "section": "Bug Reports",
        "text": "Report product bugs via the support portal. Include logs, steps to reproduce, and system information. Priority P1 is for outages; otherwise categorize as P2 or P3."
    },
]
len(sample_docs)


3

In [3]:

# BAD pattern: put everything into one giant chunk (for demonstration)
bad_chunks = [{
    "chunk_id": "all#000",
    "title": "All Docs",
    "text": "\n\n".join([f"{d['title']} — {d['section']}\n{d['text']}" for d in sample_docs])
}]
len(bad_chunks), len(bad_chunks[0]["text"])


(1, 559)

In [4]:

# GOOD pattern: recursive splitter (character-based) with overlap
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80)
good_chunks = []
for i, d in enumerate(sample_docs):
    splits = text_splitter.split_text(d["text"])
    for j, s in enumerate(splits):
        good_chunks.append({
            "chunk_id": f"{d['title'].lower().replace(' ','_')}#{j:03d}",
            "title": d["title"],
            "section": d["section"],
            "text": s
        })

len(good_chunks), [c["chunk_id"] for c in good_chunks]


(3, ['install_guide#000', 'account_faq#000', 'support_handbook#000'])


## 3) Embeddings & Vector Store (FAISS or Chroma)

- **Embed once**, **index once**, persist, and reload.
- For class speed, we’ll use **FAISS** in-memory; in projects, persist to disk (FAISS or Chroma).


In [5]:

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.docstore.document import Document

# Build Documents from good chunks
docs = [
    Document(page_content=c["text"], metadata={"chunk_id": c["chunk_id"], "title": c["title"], "section": c["section"]})
    for c in good_chunks
]

emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vs = FAISS.from_documents(docs, emb)

# Persist & reload (optional demo)
INDEX_DIR = "index-faiss"
vs.save_local(INDEX_DIR)

# Reload to show persistence
vs = FAISS.load_local(INDEX_DIR, emb, allow_dangerous_deserialization=True)
retriever = vs.as_retriever(search_kwargs={"k": 4})
retriever


/var/folders/f4/sswmwmgd3bl8rxfdhchzt6h00000gn/T/ipykernel_7152/1507094092.py:11: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x31e651190>, search_kwargs={'k': 4})


## 4) Retriever (bad vs good)

- **Bad:** `k=15`, no filters, returns too many chunks → bloated prompts, slower, worse grounding.
- **Good:** **small `k` (3–5)**, add metadata filters when needed.


In [6]:

# BAD: huge k
retriever_bad = vs.as_retriever(search_kwargs={"k": 15})
bad_results = retriever_bad.get_relevant_documents("What are the hardware requirements?")
len(bad_results), [d.metadata["chunk_id"] for d in bad_results]


/var/folders/f4/sswmwmgd3bl8rxfdhchzt6h00000gn/T/ipykernel_7152/3790287532.py:3: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  bad_results = retriever_bad.get_relevant_documents("What are the hardware requirements?")


(3, ['install_guide#000', 'support_handbook#000', 'account_faq#000'])

In [7]:

# GOOD: small k
retriever_good = vs.as_retriever(search_kwargs={"k": 4})
good_results = retriever_good.get_relevant_documents("What are the hardware requirements?")
len(good_results), [d.metadata["chunk_id"] for d in good_results]


(3, ['install_guide#000', 'support_handbook#000', 'account_faq#000'])


## 5) Grounded Prompt → JSON (bad vs good)

- **Bad prompt:** no grounding, asks for prose → model may hallucinate and returns unstructured text.
- **Good prompt:** *Use ONLY provided context*, return strict JSON `{answer, sources[]}`, and say “I don’t know based on the provided context.” when context is weak.
- Keep **temperature low** (≈0.2).


In [8]:
from langchain.prompts import ChatPromptTemplate

# BAD prompt (for demonstration) — fixed quoting with triple quotes
bad_prompt = ChatPromptTemplate.from_template(
    """Answer the user's question in detail:
Question: {question}
Context:
{context}"""
)

# GOOD prompt — escape JSON braces with double {{ }} so str.format doesn't treat them as fields
good_prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Use ONLY the context below to answer.\n"
    "Return STRICT JSON with keys: answer (string), sources (array of chunk_id strings).\n"
    "If the context is insufficient, return: {{\"answer\": \"I don't know based on the provided context.\", \"sources\": []}}"
    "\n---\n"
    "Question: {question}\n"
    "Context:\n{context}"
)
good_prompt


ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='You are a helpful assistant. Use ONLY the context below to answer.\nReturn STRICT JSON with keys: answer (string), sources (array of chunk_id strings).\nIf the context is insufficient, return: {{"answer": "I don\'t know based on the provided context.", "sources": []}}\n---\nQuestion: {question}\nContext:\n{context}'), additional_kwargs={})])


## 6) Model, Parsing & Validation

We’ll support two paths:
- **Mock LLM** for environments without keys — it returns a predictable JSON string if a known keyword appears.
- **Real LLM** with low temperature and output token cap.

We’ll parse the model’s text to JSON, **validate** required fields, **retry once** with stricter instructions if parsing fails, else show a **friendly error**.


In [9]:

import json, re, time
from langchain.schema.runnable import RunnableLambda, RunnableParallel, RunnableBranch
from langchain.output_parsers.json import SimpleJsonOutputParser

def format_context(docs):
    parts = []
    for d in docs:
        cid = d.metadata.get("chunk_id")
        title = d.metadata.get("title")
        parts.append(f"[{cid}] {title}: {d.page_content}")
    return "\n---\n".join(parts)

parser = SimpleJsonOutputParser()

# --- Mock LLM (simple demo) ---
def mock_llm_call(prompt_text: str) -> str:
    # If prompt mentions "Hardware requirements" or "install", return a matching JSON;
    # else return unstructured text to trigger the parser retry path.
    if re.search(r"8 GB RAM|hardware|install", prompt_text, re.I):
        return json.dumps({
            "answer": "You need 8 GB RAM, 20 GB free disk, and a modern CPU. GPU optional.",
            "sources": ["install_guide#000"]
        })
    return "It looks like you asked about something else; here's some thoughts..."  # not JSON

def run_model(prompt_str: str, temperature=0.2, max_tokens=256):
    if USE_REAL_LLM:
        from langchain_openai import ChatOpenAI
        llm = ChatOpenAI(model=MODEL_NAME, temperature=temperature, max_tokens=max_tokens)
        return llm.invoke(prompt_str).content
    else:
        return mock_llm_call(prompt_str)

def ask_chain(question: str, docs, prompt_tmpl: ChatPromptTemplate, retry_once: bool = True):
    ctx = format_context(docs)
    prompt_str = prompt_tmpl.format(question=question, context=ctx)
    text = run_model(prompt_str)

    try:
        data = parser.parse(text)
        assert isinstance(data.get("sources"), list)
        assert data.get("answer") is not None
        return data, {"parse_ok": True, "raw": text}
    except Exception as e:
        if retry_once:
            # Retry with stricter format reminder
            strict_prompt = prompt_str + "\n\nReturn ONLY valid JSON with keys: answer, sources."
            text2 = run_model(strict_prompt)
            try:
                data = parser.parse(text2)
                assert isinstance(data.get("sources"), list)
                assert data.get("answer") is not None
                return data, {"parse_ok": True, "raw": text2, "retried": True}
            except Exception:
                pass
        # Friendly error
        return {"answer": None, "sources": []}, {"parse_ok": False, "raw": text}

# Try BAD prompt (expect parse failure), then GOOD prompt (parse OK)
docs_for_q = retriever_good.get_relevant_documents("What are the hardware requirements?")
bad_result = run_model(bad_prompt.format(question="What are the hardware requirements?", context=format_context(docs_for_q)))
good_result, meta = ask_chain("What are the hardware requirements?", docs_for_q, good_prompt)

print("BAD prompt raw:", bad_result[:120], "...")
print("GOOD prompt parsed:", good_result, "| meta:", meta)


BAD prompt raw: {"answer": "You need 8 GB RAM, 20 GB free disk, and a modern CPU. GPU optional.", "sources": ["install_guide#000"]} ...
GOOD prompt parsed: {'answer': 'You need 8 GB RAM, 20 GB free disk, and a modern CPU. GPU optional.', 'sources': ['install_guide#000']} | meta: {'parse_ok': True, 'raw': '{"answer": "You need 8 GB RAM, 20 GB free disk, and a modern CPU. GPU optional.", "sources": ["install_guide#000"]}'}



## 7) Guardrails: ACL filter, PII scrub, context size

- **ACL**: filter docs by `tenant_id`/`role` **before** prompting (vector store filter or post-filter for FAISS).
- **PII scrub**: mask emails/IDs in retrieved text *before* the prompt.
- **Context size**: keep `k` small; optionally use a **compression retriever**.

Below we simulate ACL and scrubbing inline.


In [10]:

# Simulate ACL: add a tenant flag to metadata and filter
for i, d in enumerate(docs):
    d.metadata["tenant_id"] = "tenant_A" if i % 2 == 0 else "tenant_B"

def acl_filter(doc_list, allow_tenant="tenant_A"):
    return [d for d in doc_list if d.metadata.get("tenant_id")==allow_tenant]

import re
def scrub_text(doc_list):
    scrubbed = []
    for d in doc_list:
        t = d.page_content
        t = re.sub(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", "[email]", t)
        t = re.sub(r"\b[A-Z0-9]{8,}\b", "[id]", t)
        d2 = Document(page_content=t, metadata=d.metadata)
        scrubbed.append(d2)
    return scrubbed

q = "Where do I report bugs?"
docs_raw = retriever_good.get_relevant_documents(q)
docs_acl = acl_filter(docs_raw, allow_tenant="tenant_A")
docs_clean = scrub_text(docs_acl)

good_result, meta = ask_chain(q, docs_clean, good_prompt)
good_result, meta


({'answer': None, 'sources': []},
 {'parse_ok': False,
  'raw': "It looks like you asked about something else; here's some thoughts..."})


## 8) Tools (optional, add only when it clearly helps)

We’ll add one **safe calculator** tool and sketch a simple routing rule.

> In a live app, also consider `TavilySearchResults` or `DuckDuckGoSearchRun` for web lookups, with **allow-lists** and **timeouts**.


In [11]:
# Expected issue, the math tool only accepts math operators, not the full question, this is a classical mistake people make:
from langchain.tools import tool

@tool("safe_calculator")
def safe_calculator(expression: str) -> str:
    """Evaluate simple arithmetic like 12 * 7. No variables/functions."""
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        raise ValueError("Unsupported characters")
    return str(eval(expression))

def maybe_call_calc(question: str):
    if re.search(r"\d\s*[\+\-\*/]\s*\d", question):
        return safe_calculator.run(question)
    return None

def ask_chain_with_tool(question: str, docs, prompt_tmpl: ChatPromptTemplate):
    tool_result = maybe_call_calc(question)
    ctx = format_context(docs)
    if tool_result is not None:
        ctx = ctx + f"\n---\n[tool:safe_calculator] result: {tool_result}"
    prompt_str = prompt_tmpl.format(question=question, context=ctx)
    text = run_model(prompt_str)
    try:
        data = parser.parse(text)
    except Exception:
        data = {"answer": None, "sources": []}
    # If a tool contributed, add a synthetic source tag
    if tool_result is not None:
        data["sources"] = list(set(data.get("sources", []) + ["tool:safe_calculator"]))
    return data

ask_chain_with_tool("What is 12 * 7?", docs_clean, good_prompt)


ValueError: Unsupported characters

In [12]:
# Patch: extract only the arithmetic expression before calling the tool
import re

# Accepts: 12*7, 12 * 7, 3+4-5/2, (2+3)*4, 1.5 * 2
_ARITH_RE = re.compile(r'(?P<expr>\d+(?:\.\d+)?(?:\s*[\+\-\*/]\s*\(?\d+(?:\.\d+)?\)?)+)')

def maybe_call_calc(question: str):
    m = _ARITH_RE.search(question)
    if not m:
        return None
    expr = m.group("expr")                 # only the math part, e.g. "12 * 7"
    return safe_calculator.run(expr)

def ask_chain_with_tool(question: str, docs, prompt_tmpl: ChatPromptTemplate):
    tool_result = maybe_call_calc(question)
    ctx = format_context(docs)
    if tool_result is not None:
        ctx += f'\n---\n[tool:safe_calculator] result: {tool_result}'
    prompt_str = prompt_tmpl.format(question=question, context=ctx)
    text = run_model(prompt_str)
    try:
        data = parser.parse(text)
    except Exception:
        data = {"answer": None, "sources": []}
    if tool_result is not None:
        data["sources"] = list(set(data.get("sources", []) + ["tool:safe_calculator"]))
    return data

# Re-run: now this should work and add "tool:safe_calculator" to sources
ask_chain_with_tool("What is 12 * 7?", docs_clean, good_prompt)


{'answer': None, 'sources': ['tool:safe_calculator']}


## 9) Short Memory (opt-in, safe)

Use `RunnableWithMessageHistory` and a **short window** (e.g., last 3 user turns). Memory helps with **follow-ups** but does not replace retrieval.


In [13]:
# Compat import for different LangChain versions
try:
    from langchain_core.runnables.history import RunnableWithMessageHistory
except ImportError:  # older versions
    from langchain.schema.runnable import RunnableWithMessageHistory  # type: ignore


In [14]:

from langchain.memory import ChatMessageHistory
from langchain.schema import messages_from_dict, messages_to_dict



store = {}
def get_history(session_id):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    # keep short window
    store[session_id].messages = store[session_id].messages[-6:]
    return store[session_id]

# We'll wrap only the question pathway; our demo chain is a function,
# so we simulate memory by recording messages and passing a 'history' string.
def simulate_memory(session_id: str, user_text: str, docs):
    hist = get_history(session_id)
    hist.add_user_message(user_text)
    # (In a full LCEL, you'd include MessagesPlaceholder('history') in the prompt)
    data = ask_chain(user_text, docs, good_prompt)[0]
    hist.add_ai_message(data.get("answer") or "(no answer)")
    return data

session_id = "demo_user_1"
simulate_memory(session_id, "Tell me the install requirements", docs_clean)
simulate_memory(session_id, "Repeat step 2 please", docs_clean)  # relies on short memory in real chain


{'answer': None, 'sources': []}


## 10) Observe (LangSmith)

LangSmith captures **runs/spans** (retriever → prompt → model → parser), **tokens**, **latency**, and **parse_ok**. It also supports small **datasets** and quick **evaluations**.

Enable with:
```python
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Module9_RAG"]
```
Then run your chain as usual; inspect in the LangSmith UI.



## 11) Serve (LangServe + FastAPI)

**Contract stays the same**: request `{question}` → response `{answer, sources[]}`

```python
# server.py
from fastapi import FastAPI
from langserve import add_routes

app = FastAPI(title="Module9 RAG")
# 'chain' should be your LCEL pipeline that returns {"answer":..., "sources":[...]}
add_routes(app, chain, path="/rag")

# run: uvicorn server:app --reload --port 8000
```



## 12) UI (Gradio)

A tiny chat that calls `/rag/invoke` and prints the answer with **source chips**:
```python
# ui.py
import gradio as gr, requests

API = "http://localhost:8000/rag/invoke"
def ask(q):
    r = requests.post(API, json={"input": {"question": q}}, timeout=10).json()
    out = r.get("output", {})
    chips = " ".join(f"[{s}]" for s in out.get("sources", []))
    return f"{out.get('answer', '(no answer)')}\n\nSources: {chips}"

gr.ChatInterface(ask, title="Module 9 — RAG Chat").launch()
```



---

### Wrap-up
You now have a complete, **grounded** RAG path: load/split → embed/index → retrieve → **grounded prompt to strict JSON** → parse/validate → guardrails → (optional) one tool → short memory → observe and serve with the **same contract**.

> For Lesson 4 (LlamaIndex), you can repeat the same demo shape—focus on loaders, indices, retrievers, and the same JSON contract and guardrails—to compare frameworks by theme.
